# Project 4 — Molecular Classification

This notebook converts molecular solubility into a binary classification task. Instead of predicting exact `target_logS`, the model predicts whether a molecule belongs to a higher-solubility or lower-solubility class.

Classification changes the whole evaluation logic. Accuracy alone is not enough; confusion matrices, precision, recall, F1, ROC-AUC, PR-AUC and threshold tuning become important.

The notebook uses public solubility data when available, an offline fallback dataset when needed and a 500-molecule cap for safe execution.

## 1. Setup

This notebook uses RDKit to calculate molecular features and scikit-learn to train classification models. The metrics are different from regression because the output is a class label or class probability.

Classification pipelines should keep preprocessing and model fitting together, especially when scaling is required.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, Descriptors, Crippen, Lipinski, rdMolDescriptors
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, RocCurveDisplay, PrecisionRecallDisplay,
    roc_curve, precision_recall_curve,
)

## 2. Load molecules and create labels

The continuous solubility target is converted into a binary label using a threshold. A median split is useful for a balanced educational example, although scientific projects may need a domain-specific threshold.

The label design defines the question being asked. Changing the threshold changes the biological or chemical interpretation of the classifier.

In [ ]:
def make_demo_solubility_data():
    records = [
        {"name": "methanol", "smiles": "CO", "target_logS": 0.10},
        {"name": "ethanol", "smiles": "CCO", "target_logS": -0.20},
        {"name": "propanol", "smiles": "CCCO", "target_logS": -0.70},
        {"name": "butanol", "smiles": "CCCCO", "target_logS": -1.10},
        {"name": "pentanol", "smiles": "CCCCCO", "target_logS": -1.70},
        {"name": "hexanol", "smiles": "CCCCCCO", "target_logS": -2.20},
        {"name": "acetone", "smiles": "CC(=O)C", "target_logS": -0.20},
        {"name": "acetic acid", "smiles": "CC(=O)O", "target_logS": 0.10},
        {"name": "ethyl acetate", "smiles": "CCOC(=O)C", "target_logS": -0.70},
        {"name": "diethyl ether", "smiles": "CCOCC", "target_logS": -1.00},
        {"name": "benzene", "smiles": "c1ccccc1", "target_logS": -2.10},
        {"name": "toluene", "smiles": "Cc1ccccc1", "target_logS": -2.70},
        {"name": "phenol", "smiles": "Oc1ccccc1", "target_logS": -1.50},
        {"name": "aniline", "smiles": "Nc1ccccc1", "target_logS": -1.30},
        {"name": "benzoic acid", "smiles": "O=C(O)c1ccccc1", "target_logS": -1.80},
        {"name": "benzaldehyde", "smiles": "O=Cc1ccccc1", "target_logS": -1.90},
        {"name": "benzyl alcohol", "smiles": "OCc1ccccc1", "target_logS": -1.10},
        {"name": "vanillin", "smiles": "COc1cc(C=O)ccc1O", "target_logS": -1.20},
        {"name": "cinnamaldehyde", "smiles": "O=CC=Cc1ccccc1", "target_logS": -2.30},
        {"name": "limonene", "smiles": "CC1=CCC(CC1)C(=C)C", "target_logS": -4.30},
        {"name": "linalool", "smiles": "CC(C)=CCCC(C)(O)C=C", "target_logS": -2.80},
        {"name": "menthol", "smiles": "CC(C)C1CCC(C)CC1O", "target_logS": -3.20},
        {"name": "aspirin", "smiles": "CC(=O)OC1=CC=CC=C1C(=O)O", "target_logS": -2.30},
        {"name": "caffeine", "smiles": "Cn1cnc2c1c(=O)n(C)c(=O)n2C", "target_logS": -1.00},
        {"name": "paracetamol", "smiles": "CC(=O)NC1=CC=C(O)C=C1", "target_logS": -1.40},
        {"name": "ibuprofen", "smiles": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O", "target_logS": -4.00},
        {"name": "naphthalene", "smiles": "c1ccc2ccccc2c1", "target_logS": -3.60},
        {"name": "anthracene", "smiles": "c1ccc2cc3ccccc3cc2c1", "target_logS": -5.10},
        {"name": "glucose", "smiles": "C(C1C(C(C(C(O1)O)O)O)O)O", "target_logS": 0.70},
        {"name": "cholesterol", "smiles": "CC(C)CCCC(C)C1CCC2C3CCC4=CC(O)CCC4(C)C3CCC12C", "target_logS": -7.00},
    ]
    return pd.DataFrame(records)


def standardise_solubility_table(raw_df):
    column_lookup = {col.lower(): col for col in raw_df.columns}

    smiles_col = column_lookup.get("smiles")
    target_col = column_lookup.get("solubility") or column_lookup.get("target_logs") or column_lookup.get("logs")
    name_col = column_lookup.get("name")

    if smiles_col is None or target_col is None:
        raise ValueError("The public dataset does not contain the expected SMILES and solubility columns.")

    names = raw_df[name_col] if name_col is not None else [f"molecule_{i}" for i in range(len(raw_df))]

    # Keep only the columns needed by the downstream notebook.
    df = pd.DataFrame({
        "name": names,
        "smiles": raw_df[smiles_col],
        "target_logS": raw_df[target_col],
    })

    # Remove incomplete records before modelling.
    df = df.dropna(subset=["smiles", "target_logS"]).copy()

    # Convert the target to numeric LogS values.
    df["target_logS"] = pd.to_numeric(df["target_logS"], errors="coerce")
    df = df.dropna(subset=["target_logS"]).copy()

    return df


def load_aqsol_public_data(timeout_seconds=10):
    from urllib.request import Request, urlopen

    url = "https://raw.githubusercontent.com/mcsorkun/AqSolDB/master/results/data_curated.csv"
    request = Request(url, headers={"User-Agent": "chem-ml-practice-notebook"})

    # Use a network timeout so offline notebooks fall back instead of hanging.
    with urlopen(request, timeout=timeout_seconds) as response:
        raw_df = pd.read_csv(response, low_memory=False)

    return standardise_solubility_table(raw_df)


def load_solubility_data(max_molecules=500, random_state=42):
    try:
        df = load_aqsol_public_data()
        data_source = "AqSolDB public dataset"
    except Exception as error:
        print("Public dataset loading failed:", error)
        df = make_demo_solubility_data()
        data_source = "built-in demo dataset"

    # Randomly cap the dataset before descriptor/fingerprint calculation.
    if len(df) > max_molecules:
        df = df.sample(n=max_molecules, random_state=random_state).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    return df, data_source

MAX_MOLECULES = 500
RANDOM_STATE = 42

df, data_source = load_solubility_data(max_molecules=MAX_MOLECULES, random_state=RANDOM_STATE)

# Convert SMILES into molecule objects after the 500-row cap.
df["mol"] = df["smiles"].apply(lambda smiles: Chem.MolFromSmiles(str(smiles)))

# Keep rows with valid molecule objects.
df = df[df["mol"].notnull()].copy().reset_index(drop=True)

# Median split gives a balanced binary label for classification practice.
threshold = df["target_logS"].median()
df["label"] = (df["target_logS"] > threshold).astype(int)

print("Data source:", data_source)
print("Dataset used:", df.shape)
print("Median logS threshold:", threshold)
print(df["label"].value_counts().sort_index())
df[["name", "smiles", "target_logS", "label"]].head()


### Exercise — compare different label thresholds

Create labels using at least three thresholds: lower quartile, median and upper quartile of `target_logS`. For each threshold, count class 0 and class 1 molecules.

Guidance: label balance strongly affects classification. This exercise shows why the threshold should be chosen deliberately rather than treated as a technical detail.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Calculate three target thresholds.


    # For each threshold, create labels and count class sizes.


    # Store the counts in a dataframe.


    pass
else:
    print("Set RUN_EXERCISE = True after writing the threshold comparison.")


### Answer — compare different label thresholds

Reference solution.

In [ ]:
thresholds = {
    "lower_quartile": df["target_logS"].quantile(0.25),
    "median": df["target_logS"].median(),
    "upper_quartile": df["target_logS"].quantile(0.75),
}

threshold_rows = []
for threshold_name, threshold_value in thresholds.items():
    temp_label = (df["target_logS"] >= threshold_value).astype(int)
    counts = temp_label.value_counts().to_dict()
    threshold_rows.append({
        "threshold_name": threshold_name,
        "threshold_value": threshold_value,
        "class_0_count": counts.get(0, 0),
        "class_1_count": counts.get(1, 0),
    })

threshold_comparison_df = pd.DataFrame(threshold_rows)
threshold_comparison_df


## 3. Build classification features

The feature table uses interpretable RDKit descriptors. Descriptor features make it easier to inspect how classes differ before fitting classifiers.

Class-wise descriptor summaries can reveal whether the label is associated with lipophilicity, polarity, molecular size or hydrogen bonding.

In [ ]:
def calculate_classification_descriptors(mol):
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "RingCount": Descriptors.RingCount(mol),
    }

# Calculate descriptor features for each molecule.
descriptor_df = df["mol"].apply(calculate_classification_descriptors).apply(pd.Series)

# Use descriptor columns as classification inputs.
feature_columns = descriptor_df.columns.tolist()
data_df = pd.concat([df.reset_index(drop=True), descriptor_df], axis=1)

X = data_df[feature_columns]
y = data_df["label"]

print("Feature matrix:", X.shape)
data_df[["name", "label"] + feature_columns].head()

### Exercise — inspect feature separation between classes

Create a table that compares mean descriptor values between class 0 and class 1. Add a difference column showing class 1 mean minus class 0 mean, then sort by absolute difference.

Guidance: this helps identify which descriptors may be useful for classification before fitting any model.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Calculate class-wise descriptor means.


    # Create a difference column comparing class 1 against class 0.


    # Sort descriptors by absolute difference.


    pass
else:
    print("Set RUN_EXERCISE = True after building the feature-separation table.")


### Answer — inspect feature separation between classes

Reference solution.

In [ ]:
class_means = data_df.groupby("label")[feature_columns].mean().T
class_means.columns = [f"class_{col}_mean" for col in class_means.columns]

if {"class_0_mean", "class_1_mean"}.issubset(class_means.columns):
    class_means["difference_class1_minus_class0"] = class_means["class_1_mean"] - class_means["class_0_mean"]
    class_means["absolute_difference"] = class_means["difference_class1_minus_class0"].abs()

class_means.sort_values("absolute_difference", ascending=False)


## 4. Train classifiers

Logistic regression gives a simple linear probability baseline. Random Forest and Extra Trees provide non-linear tree-based baselines.

A useful comparison uses the same train/test split for every model and records both class-label metrics and probability-based metrics.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

classifiers = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000)),
    ]),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=100, random_state=42),
}

rows = []
fitted_classifiers = {}

for model_name, model in classifiers.items():
    # Fit the classifier on training molecules.
    model.fit(X_train, y_train)
    fitted_classifiers[model_name] = model

    # Predict hard labels and class-1 probabilities.
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    rows.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "pr_auc": average_precision_score(y_test, y_prob),
    })

result_df = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
result_df

### Exercise — build a classifier benchmark table

Train at least three classifiers and collect accuracy, precision, recall, F1, ROC-AUC and PR-AUC. Use predicted probabilities for ROC-AUC and PR-AUC.

Guidance: classification models should be compared with several metrics because each metric highlights a different type of error.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Define at least three classifiers.


    # Fit each classifier on the training set.


    # Calculate class-label metrics and probability metrics.


    # Store the results in a dataframe.


    pass
else:
    print("Set RUN_EXERCISE = True after writing your classifier benchmark.")


### Answer — build a classifier benchmark table

Reference solution.

In [ ]:
custom_classifiers = {
    "LogisticRegression": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=200, random_state=42),
}

benchmark_rows = []
for model_name, model in custom_classifiers.items():
    model.fit(X_train, y_train)
    pred_label = model.predict(X_test)
    pred_prob = model.predict_proba(X_test)[:, 1]
    benchmark_rows.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, pred_label),
        "precision": precision_score(y_test, pred_label, zero_division=0),
        "recall": recall_score(y_test, pred_label, zero_division=0),
        "F1": f1_score(y_test, pred_label, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, pred_prob),
        "PR_AUC": average_precision_score(y_test, pred_prob),
    })

classifier_benchmark_df = pd.DataFrame(benchmark_rows).sort_values("F1", ascending=False)
classifier_benchmark_df


## 5. Confusion matrix and classification report

The confusion matrix shows how predictions break down into true positives, true negatives, false positives and false negatives. This is more informative than accuracy alone.

Precision and recall answer different questions. Precision asks how many predicted positives were correct; recall asks how many true positives were recovered.

In [ ]:
best_model_name = result_df.iloc[0]["model"]
best_model = fitted_classifiers[best_model_name]

# Predict labels and probabilities with the best classifier.
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Best model:", best_model_name)
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print("Classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

### Exercise — manually interpret confusion-matrix errors

Choose the best classifier from the benchmark, calculate its confusion matrix and create a small table explaining TP, TN, FP and FN counts in plain terms.

Guidance: this exercise connects metric output to actual prediction behaviour. It also prepares the ground for threshold tuning.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Select a fitted classifier.


    # Calculate the confusion matrix.


    # Convert TP, TN, FP and FN into a readable table.


    pass
else:
    print("Set RUN_EXERCISE = True after writing the confusion-matrix interpretation.")


### Answer — manually interpret confusion-matrix errors

Reference solution.

In [ ]:
best_classifier_name = classifier_benchmark_df.iloc[0]["model"]
best_classifier = custom_classifiers[best_classifier_name]
best_classifier.fit(X_train, y_train)

best_pred_label = best_classifier.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, best_pred_label).ravel()

confusion_summary = pd.DataFrame([
    {"cell": "TN", "count": tn, "meaning": "actual class 0 predicted as class 0"},
    {"cell": "FP", "count": fp, "meaning": "actual class 0 predicted as class 1"},
    {"cell": "FN", "count": fn, "meaning": "actual class 1 predicted as class 0"},
    {"cell": "TP", "count": tp, "meaning": "actual class 1 predicted as class 1"},
])

confusion_summary


## 6. ROC and precision-recall curves

ROC curves describe the trade-off between true-positive rate and false-positive rate across thresholds. Precision-recall curves focus on the positive class and are often more informative when classes are imbalanced.

These curves use predicted probabilities, not just hard class labels.

In [ ]:
# Plot ROC curve from class-1 probabilities.
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title(f"ROC curve: {best_model_name}")
plt.show()

In [ ]:
# Plot precision-recall curve from class-1 probabilities.
PrecisionRecallDisplay.from_predictions(y_test, y_prob)
plt.title(f"Precision-recall curve: {best_model_name}")
plt.show()

### Exercise — plot ROC and precision-recall curves for two models

Select two classifiers and plot their ROC curves and precision-recall curves. Compare whether the same model looks best under both curve types.

Guidance: ROC-AUC and PR-AUC can rank models differently, especially when class balance changes.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Choose two fitted classifiers.


    # Plot ROC curves using predicted probabilities.


    # Plot precision-recall curves using predicted probabilities.


    pass
else:
    print("Set RUN_EXERCISE = True after writing the ROC and PR curve comparison.")


### Answer — plot ROC and precision-recall curves for two models

Reference solution.

In [ ]:
selected_models = {
    "LogisticRegression": custom_classifiers["LogisticRegression"],
    "RandomForest": custom_classifiers["RandomForest"],
}

plt.figure(figsize=(7, 5))
for model_name, model in selected_models.items():
    model.fit(X_train, y_train)
    pred_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, pred_prob)
    plt.plot(fpr, tpr, label=model_name)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves")
plt.legend()
plt.show()

plt.figure(figsize=(7, 5))
for model_name, model in selected_models.items():
    pred_prob = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, pred_prob)
    plt.plot(recall, precision, label=model_name)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall curves")
plt.legend()
plt.show()


## 7. Tune the classification threshold

The default threshold of 0.5 is not always optimal. Lowering or raising the probability threshold changes precision, recall and F1.

Threshold tuning makes the classifier more flexible because different applications may prefer fewer false positives or fewer false negatives.

In [ ]:
threshold_rows = []
for cutoff in np.linspace(0.2, 0.8, 7):
    # Convert probabilities into labels using the current cutoff.
    tuned_pred = (y_prob >= cutoff).astype(int)

    threshold_rows.append({
        "threshold": cutoff,
        "precision": precision_score(y_test, tuned_pred, zero_division=0),
        "recall": recall_score(y_test, tuned_pred, zero_division=0),
        "f1": f1_score(y_test, tuned_pred, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df

### Exercise — choose a threshold for a chosen objective

Use predicted probabilities from one classifier and test thresholds from 0.10 to 0.90. Build a table with precision, recall and F1 for each threshold, then choose the threshold that best matches your objective.

Guidance: high recall and high precision are not always achieved at the same threshold. The threshold should reflect the type of error you want to minimise.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Generate threshold values from 0.10 to 0.90.


    # Convert probabilities into labels at each threshold.


    # Calculate precision, recall and F1.


    # Choose a threshold based on your objective.


    pass
else:
    print("Set RUN_EXERCISE = True after writing the threshold search.")


### Answer — choose a threshold for a chosen objective

Reference solution.

In [ ]:
probabilities = best_classifier.predict_proba(X_test)[:, 1]
threshold_rows = []
for threshold_value in np.linspace(0.10, 0.90, 17):
    threshold_pred = (probabilities >= threshold_value).astype(int)
    threshold_rows.append({
        "threshold": threshold_value,
        "precision": precision_score(y_test, threshold_pred, zero_division=0),
        "recall": recall_score(y_test, threshold_pred, zero_division=0),
        "F1": f1_score(y_test, threshold_pred, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.sort_values("F1", ascending=False).head(10)


## 8. Inspect misclassified molecules

Misclassified molecules show where the classifier struggles. They may sit near the threshold, have unusual descriptor patterns or belong to chemical regions that are under-represented.

Inspecting names, structures, descriptors and predicted probabilities is often more useful than only reporting aggregate metrics.

In [ ]:
test_indices = X_test.index

prediction_df = data_df.loc[test_indices, ["name", "smiles", "target_logS", "label"] + feature_columns].copy()
prediction_df["predicted_label"] = y_pred
prediction_df["prob_high_solubility"] = y_prob
prediction_df["is_correct"] = prediction_df["label"] == prediction_df["predicted_label"]

prediction_df.sort_values("prob_high_solubility", ascending=False)

In [ ]:
# Select molecules where the predicted class differs from the true class.
misclassified_df = prediction_df[~prediction_df["is_correct"]]

# Draw misclassified molecules if any exist.
if len(misclassified_df) > 0:
    misclassified_mols = [Chem.MolFromSmiles(smiles) for smiles in misclassified_df["smiles"]]
    display(Draw.MolsToGridImage(misclassified_mols, legends=misclassified_df["name"].tolist(), molsPerRow=3))
else:
    print("No misclassified molecules in this split.")

### Exercise — build a misclassification review table

Create a table of test-set molecules with actual label, predicted label, predicted probability and key descriptors. Filter to misclassified molecules and sort by confidence.

Guidance: confident mistakes are especially important because they reveal where a model is not only wrong, but wrong with high certainty.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Predict labels and probabilities for the test set.


    # Build a review table with molecule identity, descriptors and prediction results.


    # Filter to misclassified molecules and sort by confidence.


    pass
else:
    print("Set RUN_EXERCISE = True after building the misclassification review table.")


### Answer — build a misclassification review table

Reference solution.

In [ ]:
review_prob = best_classifier.predict_proba(X_test)[:, 1]
review_pred = (review_prob >= 0.5).astype(int)

review_df = data_df.loc[X_test.index, ["name", "smiles", "target_logS", "label", "MolWt", "LogP", "TPSA"]].copy()
review_df["predicted_label"] = review_pred
review_df["predicted_probability"] = review_prob
review_df["is_correct"] = review_df["label"] == review_df["predicted_label"]
review_df["confidence"] = np.where(review_df["predicted_label"] == 1, review_prob, 1 - review_prob)

misclassified_review_df = review_df[~review_df["is_correct"]].sort_values("confidence", ascending=False)
misclassified_review_df.head(10)
